# 🦖 Example 6: Deep-Time Paleo-Globes & Plate Reconstructions

This tutorial explains how to generate 3D printable globes representing ancient Earth across deep geological time (0 to 540 Ma) using plate tectonic reconstructions, Scotese PALEOMAP PaleoDEM elevation grids, full-color paleogeographic textures, and modern location tracking.

### 🌎 Key Capabilities Demonstrated:
1. **Paleo Datasets**: Loading Scotese & Wright (2018) PaleoDEMs and matching PaleoAtlas textures via `globe3d.datasets.paleo`.
2. **Deep-Time Feature Tracking**: Tracking modern landmarks (e.g., Oxford, UK or Baltimore, USA) back to their paleo-positions across continental drift using `PlateModel`.
3. **Spherical Vector Symbol Stamping**: Projecting SVG star logos and pole orientation markers ('N'/'S') onto the globe texture without polar distortion.
4. **Coastline Shelf Step**: Accentuating ancient shorelines with synthetic vertical exaggeration.
5. **Texture Preprocessing for 3D Printing**: Eliminating hillshade shadows and quantizing colors to a filament palette.
6. **One-Line Factory & Modular Pipeline**: Using `GlobeModel.from_paleo` or assembling custom components.

In [ ]:
import numpy as np
from PIL import Image
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    ImageColourer,
    PlateModel,
    reconstruct_points,
    datasets,
    add_svg_symbol_spherical,
    add_pole_markers,
    undo_hillshading,
    simplify_colors,
    calculate_displacement_scale,
)

## 1. Inspecting the Scotese PALEOMAP Catalog
`globe3d` includes a registry of 108 geological time slices from the Holocene (0 Ma) to the Late Proterozoic / Cambrian (540 Ma).

In [ ]:
maps = datasets.paleo.list_scotese_maps()
print(f"Total available time slices: {len(maps)}")
for m in maps[:10]:
    print(f"{m['age_ma']:>5.1f} Ma: Map {m['map_id']:02d} — {m['name']}")

## 2. Reconstructing Modern Coordinates Across Deep Time
Using the PALEOMAP Global Plate Model, we can trace where modern cities were located millions of years ago.
For example, let's track **Oxford, UK** (51.75°N, -1.26°E) at the **Permo-Triassic Boundary (250 Ma)** when Pangea was assembled:

In [ ]:
pm = datasets.paleo.scotese_plate_model()

oxford_modern = [51.7519, -1.2578]
baltimore_modern = [39.2904, -76.6122]

oxford_250ma = pm.reconstruct_points(oxford_modern, age_ma=250.0)[0]
baltimore_250ma = pm.reconstruct_points(baltimore_modern, age_ma=250.0)[0]

print(f"Oxford at 250 Ma:    {oxford_250ma[0]:.2f}°N, {oxford_250ma[1]:.2f}°E (Subtropical Pangea interior)")
print(f"Baltimore at 250 Ma: {baltimore_250ma[0]:.2f}°N, {baltimore_250ma[1]:.2f}°E (Equatorial Pangaean collision)")

## 3. High-Level Creation with `GlobeModel.from_paleo`
The high-level `GlobeModel.from_paleo` factory configures everything in a single call:
- Loads the closest PaleoDEM and matching texture
- Applies a `coastline_step` to elevate continental landmasses
- Reconstructs and stamps tracked locations
- Adds North/South pole markers
- Hollows the sphere for 3D printing

In [ ]:
model = GlobeModel.from_paleo(
    age=250.0,  # Permo-Triassic Boundary
    radius=100.0,  # 100 mm radius (200 mm diameter)
    n_points=25000,  # Mesh resolution
    hollow=True,
    inner_ratio=0.9,
    vert_exagg=40.0,
    coastline_step=1000.0,  # 1000m shelf step to make coastlines pop
    mark_poles=True,
    tracked_locations=[
        {'lat': 51.7519, 'lon': -1.2578, 'name': 'Oxford', 'symbol': 'star', 'size': 2.5, 'color': (255, 0, 0)},
        {'lat': 39.2904, 'lon': -76.6122, 'name': 'Baltimore', 'symbol': 'star', 'size': 2.5, 'color': (255, 255, 0)},
    ]
)

print("Created GlobeModel for 250 Ma!")
print(f"Outer shell: {len(model.outer.vertices)} vertices")
print(f"Inner void:  {len(model.inner.vertices)} vertices")

## 4. Multi-Material Filament Palette Quantization
When preparing models for multi-color 3D printing (e.g. Bambu AMS or Prusa MMU), we can simplify continuous map textures to a small discrete set of filament colors:

In [ ]:
filament_palette = [
    (24, 66, 104),    # Deep ocean blue
    (142, 191, 206),  # Shallow sea cyan
    (43, 67, 46),     # Lowland green
    (184, 122, 62),   # Highland brown / desert
    (255, 0, 0),      # Marker red
]

print("Filament palette defined:", len(filament_palette), "materials")

## 5. Splitting Hemispheres & Exporting
Just like modern globes, paleo-models can be hollowed, configured with equatorial magnet voids, and exported into 3D printable hemisphere files:

In [ ]:
# Split and export (uncomment to save to disk)
# model.export_hemispheres(
#     upper_path='permo_triassic_250ma_upper.obj',
#     lower_path='permo_triassic_250ma_lower.obj',
#     flip_lower=True,
# )
print("Paleo-globe pipeline complete!")